# AMPR Phase 3 — Precompute ESM-2 + DeepGO PPI on Colab T4

Run this notebook on **Google Colab with T4 GPU** (~8-12h total, resumable).

Prerequisites in your Google Drive:
- `ampr/graph_new_embeddings.pkl` — DeepGO PPI embeddings (256d)
- `ampr/pdb_chain_uniprot.csv` — SIFTS PDB-chain → UniProt mapping
- `kaggle.json` — Kaggle API credentials

Outputs uploaded to Kaggle Dataset `ampr-phase3-embeddings`:
- `esm2_residue.h5` — per-residue ESM-2 650M embeddings (36,641 proteins)
- `ppi_deepgo.npy` — DeepGO PPI 256d embeddings
- `ppi_deepgo_mask.npy` — boolean mask (True = has PPI embedding)
- `ppi_coverage.json` — coverage statistics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/YOUR_USERNAME/datn /content/datn
%cd /content/datn
!pip install -q transformers==4.41.2 biopython==1.84 h5py tqdm pyyaml

In [ ]:
import os
os.makedirs('data/external', exist_ok=True)
os.makedirs('data/embeddings', exist_ok=True)
!ln -sf /content/drive/MyDrive/ampr/graph_new_embeddings.pkl data/external/graph.pkl
!ln -sf /content/drive/MyDrive/ampr/pdb_chain_uniprot.csv data/external/sifts.csv
print('Symlinks created.')

In [ ]:
# ESM-2 precompute (resumable — skips already-done proteins)
!python scripts/precompute_esm2_residue.py \
  --fasta data/pdbch/nrPDB-GO_2019.06.18_sequences.fasta \
  --protein_order data/pdbch/protein_order.json \
  --out data/embeddings/esm2_residue.h5 \
  --batch 4 --max_len 1022

In [ ]:
# Build DeepGO PPI embeddings + mask
!python scripts/build_ppi_from_deepgo.py \
  --pkl data/external/graph.pkl \
  --sifts data/external/sifts.csv \
  --protein_order data/pdbch/protein_order.json \
  --out_emb data/embeddings/ppi_deepgo.npy \
  --out_mask data/embeddings/ppi_deepgo_mask.npy \
  --out_coverage data/embeddings/ppi_coverage.json
!cat data/embeddings/ppi_coverage.json

In [ ]:
# Push embeddings to Kaggle Dataset
!pip install -q kaggle
import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

os.makedirs('/content/upload', exist_ok=True)
for f in ['data/embeddings/esm2_residue.h5',
          'data/embeddings/ppi_deepgo.npy',
          'data/embeddings/ppi_deepgo_mask.npy',
          'data/embeddings/ppi_coverage.json']:
    shutil.copy(f, '/content/upload/')

from datetime import datetime
msg = f'phase3 embeddings {datetime.utcnow().isoformat()}'
!kaggle datasets version -p /content/upload -m "{msg}" --dir-mode zip
print('Upload complete.')